# Revision 02 Decode Four-Condition Ratio-Matched MS-AA + PCA + Raw HTFA


This notebook loads the within-condition MS-AA fits from notebook 019 and computes:

1. **raw 700-node HTFA** cross-subject timepoint-identification accuracy;
2. **conventional node-space PCA** at the same requested normalized ratios;
3. **full spatial-AA and temporal-AA reconstructions** at ratio-matched model orders;
4. optionally, **single-archetype and top-\(m\)** MS-AA decoding for paper-style sparse analyses.

Because the four conditions have different temporal lengths, each row records its analytical
chance level \(1/T_c\). The notebook also calculates chance-corrected accuracy and the
fraction of above-chance raw-HTFA performance retained.

The top-\(m\) section defaults to the selection logic in the existing manuscript
(single-archetype ranking and evaluation on the same condition data) so that the original
figures can be reproduced. A conspicuous warning is printed because the reviewers correctly
identified this as a selection-dependence issue. Use the full-reconstruction and PCA/raw
results as the clean paragraph-condition validation; replace top-\(m\) results with a held-out
ranking analysis for the final confirmatory revision.


## Paragraph duplicate handling

The source paragraph-scrambled array contains a duplicated subject entry. Before any
normalization, fitting, PCA, raw-HTFA decoding, or MS-AA decoding, the duplicate is
removed using the same parsing rule as the original analysis:

- **700-factor HTFA:** exclude paragraph source-column index **3**;
- **other factor counts:** exclude paragraph source-column index **0**.

The retained source indices and resulting participant counts are printed explicitly so
that the four-condition pipeline can be audited.

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

from pathlib import Path

CONDITIONS = ["intact", "paragraph", "word", "rest"]

# HTFA dimensionality used for this analysis.
# Controls which duplicated paragraph entry is removed.
FACTORS = 700

# Match the parsing used in the original analysis:
#   factors == 700 -> exclude paragraph column index 3
#   otherwise      -> exclude paragraph column index 0
PARAGRAPH_DUPLICATE_INDEX_BY_FACTORS = {
    700: 3,
    "default": 0,
}

ANALYSIS_TYPES = ["spatial", "temporal"]

FIT_DIR = Path("msaa_four_condition_ratio_within_outputs_npz")
RATIO_MAP_CSV = FIT_DIR / "ratio_K_map_unique_fits.csv"
META_CSV = FIT_DIR / "four_condition_metadata.csv"

PIEMAN_MAT_PATH = "data/input/pieman_data.mat"
HYPERTOOLS_NORMALIZE_PATH = "hypertools/normalize.py"
TRIM_REST_BY = 100

OUTPUT_DIR = Path("four_condition_ratio_decoding_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Decoding
NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42

# PCA: conventional node-space PCA, reconstructed back to 700 nodes.
RUN_RAW_HTFA = True
RUN_PCA = True
RUN_AA_FULL = True

# Top-m is expensive because every archetype must be scored.
RUN_AA_TOPM = True
TOP_M_VALUES = [1, 2, 3, 5, 7, 10, 15]

# For a quick first pass, restrict top-m to selected target ratios.
# None = every fitted ratio.
TOPM_TARGET_RATIOS = [0.05, 0.075, 0.10, 0.125, 0.15, 0.20]

# Existing manuscript-compatible selection logic.
TOPM_SELECTION_MODE = "same_data"

# Append behavior
SKIP_EXISTING = True

CONDITION_KEY_ALIASES = {
    "intact": ["intact"],
    "paragraph": [
        "paragraph", "paragraph_scrambled", "paragraph-scrambled",
        "para", "paragraphscrambled"
    ],
    "word": ["word", "word_scrambled", "word-scrambled", "wordscrambled"],
    "rest": ["rest"],
}

In [ ]:
# ============================================================
# IMPORTS / DATA HELPERS
# ============================================================

%matplotlib inline

import os
import importlib.util
import numpy as np
import pandas as pd

from scipy.io import loadmat
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA

HT_NORMALIZE_AVAILABLE = False

try:
    spec = importlib.util.spec_from_file_location(
        "normalize",
        HYPERTOOLS_NORMALIZE_PATH,
    )
    ht = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ht)
    HT_NORMALIZE_AVAILABLE = True
except Exception:
    ht = None

def to_float_array(x):
    return np.asarray(x, dtype=float)

def array_cutter(data_list, cut_length):
    out = [to_float_array(x).copy() for x in data_list]
    return out if cut_length <= 0 else [x[:-cut_length] for x in out]

def coerce_TxV(x):
    x = to_float_array(x)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D array, got {x.shape}")

    T0, V0 = x.shape

    if V0 < T0:
        x = x.T

    x = x - x.mean(axis=0, keepdims=True)

    return x.astype(np.float32, copy=False)

def normalize_across_fallback(subject_list, eps=1e-8):
    subj = [coerce_TxV(x) for x in subject_list]
    stacked = np.vstack(subj)

    mu = stacked.mean(axis=0, keepdims=True)
    sd = stacked.std(axis=0, keepdims=True) + eps

    return [
        ((x - mu) / sd).astype(np.float32, copy=False)
        for x in subj
    ]

def normalize_subject_list(subject_list):
    subj = [coerce_TxV(x) for x in subject_list]

    if HT_NORMALIZE_AVAILABLE:
        try:
            return ht.normalize(subj, normalize="across")
        except Exception:
            pass

    return normalize_across_fallback(subj)

def resolve_condition_key(mat, canonical_name):
    available = [k for k in mat.keys() if not k.startswith("__")]

    def norm(s):
        return (
            str(s).lower()
            .replace("_", "")
            .replace("-", "")
            .replace(" ", "")
        )

    available_map = {norm(k): k for k in available}

    for alias in CONDITION_KEY_ALIASES[canonical_name]:
        if norm(alias) in available_map:
            return available_map[norm(alias)]

    raise KeyError(
        f"Could not resolve '{canonical_name}'. Available keys: {available}"
    )

def paragraph_duplicate_index(factors=FACTORS):
    """Return the duplicated paragraph source-column index to remove."""
    return int(
        PARAGRAPH_DUPLICATE_INDEX_BY_FACTORS.get(
            factors,
            PARAGRAPH_DUPLICATE_INDEX_BY_FACTORS["default"],
        )
    )

def extract_condition_subjects(mat, key, condition, factors=FACTORS):
    """Extract subjects, removing the duplicate paragraph entry before analysis."""
    arr = mat[key]
    all_indices = np.arange(arr.shape[1])

    if condition == "paragraph":
        duplicate_idx = paragraph_duplicate_index(factors)
        if duplicate_idx < 0 or duplicate_idx >= arr.shape[1]:
            raise IndexError(
                f"Paragraph duplicate index {duplicate_idx} is invalid for "
                f"{arr.shape[1]} paragraph entries."
            )
        keep_indices = all_indices[all_indices != duplicate_idx]
        print(
            f"Paragraph parsing: factors={factors}; excluding source column "
            f"index {duplicate_idx}; keeping {len(keep_indices)} of "
            f"{len(all_indices)} entries."
        )
    else:
        keep_indices = all_indices

    subjects = [arr[:, i][0] for i in keep_indices]
    return subjects, keep_indices

def load_condition_data():
    mat = loadmat(PIEMAN_MAT_PATH)
    out = {}
    retained_source_indices = {}

    for condition in CONDITIONS:
        key = resolve_condition_key(mat, condition)
        subjects, keep_indices = extract_condition_subjects(
            mat, key, condition, factors=FACTORS
        )
        retained_source_indices[condition] = keep_indices

        if condition == "rest" and TRIM_REST_BY > 0:
            subjects = array_cutter(subjects, TRIM_REST_BY)

        out[condition] = normalize_subject_list(subjects)

    return out, retained_source_indices

condition_data, retained_source_indices = load_condition_data()

print("\nSubject counts after paragraph duplicate removal:")
for condition in CONDITIONS:
    X0 = coerce_TxV(condition_data[condition][0])
    print(
        condition,
        "N =", len(condition_data[condition]),
        "shape =", X0.shape,
        "retained source indices =", retained_source_indices[condition],
    )

# Sanity check against metadata produced by notebook 019.
if META_CSV.exists():
    expected_meta = pd.read_csv(META_CSV)
    for condition in CONDITIONS:
        expected_n = int(
            expected_meta.loc[
                expected_meta["condition"] == condition, "N"
            ].iloc[0]
        )
        observed_n = len(condition_data[condition])
        if observed_n != expected_n:
            raise ValueError(
                f"Subject-count mismatch for {condition}: notebook 019 metadata says "
                f"N={expected_n}, but notebook 020 loaded N={observed_n}."
            )

print("\n019/020 subject-count consistency check passed.")

ratio_map = pd.read_csv(RATIO_MAP_CSV)
meta = pd.read_csv(META_CSV)

display(meta)
display(ratio_map.head(20))


In [ ]:
# ============================================================
# COMMON DECODER
# ============================================================

def normalize_reconstruction(X):
    X = np.asarray(X, dtype=float)

    return (
        X - X.mean(axis=0, keepdims=True)
    ) / (
        X.std(axis=0, keepdims=True) + 1e-8
    )

def decoder(corrs):
    out = {
        "rank": 0.0,
        "accuracy": 0.0,
        "error": 0.0,
    }

    T_c = corrs.shape[0]

    for t in range(T_c):
        decoded_ind = int(
            np.argmax(corrs[t, :])
        )

        out["error"] += (
            abs(decoded_ind - t) / T_c
        )

        out["accuracy"] += (
            decoded_ind == t
        )

        out["rank"] += np.mean(
            (
                corrs[t, :]
                <= corrs[t, t]
            ).astype(int)
        )

    for key in out:
        out[key] /= T_c

    return out

def get_xval_assignments(ndata, nfolds, rng):
    group_assignments = np.zeros(
        ndata,
        dtype=int,
    )

    groupsize = int(
        np.ceil(ndata / nfolds)
    )

    for i in range(1, nfolds):
        inds = np.arange(
            i * groupsize,
            min(
                (i + 1) * groupsize,
                ndata,
            ),
        )

        group_assignments[inds] = i

    rng.shuffle(group_assignments)

    return group_assignments

def run_timepoint_decoding(
    stack,
    nfolds=NFOLDS_DECODE,
    nreps=NREPS_DECODE,
    seed=RNG_SEED,
):
    N, T_c, D = stack.shape
    rng = np.random.default_rng(seed)

    rows = []

    for rep in range(nreps):
        fold_ids = get_xval_assignments(
            N,
            nfolds,
            rng,
        )

        for fold in range(nfolds):
            in_mask = fold_ids == fold
            out_mask = ~in_mask

            in_mean = stack[in_mask].mean(axis=0)
            out_mean = stack[out_mask].mean(axis=0)

            corrs = (
                1.0
                - cdist(
                    in_mean,
                    out_mean,
                    metric="correlation",
                )
            )

            corrs = np.nan_to_num(
                corrs,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            result = decoder(corrs)

            rows.append({
                **result,
                "rep": rep,
                "fold": fold,
            })

    return pd.DataFrame(rows)

def summarize_runs(dec):
    return {
        "mean": float(dec["accuracy"].mean()),
        "err": float(
            dec["accuracy"].std()
            / np.sqrt(max(len(dec), 1))
        ),
        "std": float(dec["accuracy"].std()),
        "count": int(len(dec)),
        "rank_mean": float(dec["rank"].mean()),
        "error_mean": float(dec["error"].mean()),
    }

In [ ]:
# ============================================================
# RAW HTFA
# ============================================================

raw_csv = OUTPUT_DIR / "raw_htfa_summary.csv"

if RUN_RAW_HTFA:

    raw_rows = []

    for condition in CONDITIONS:
        subjects = condition_data[condition]

        stack = np.stack(
            [
                normalize_reconstruction(X)
                for X in subjects
            ],
            axis=0,
        )

        dec = run_timepoint_decoding(stack)
        summary = summarize_runs(dec)

        T_c = stack.shape[1]
        chance = 1.0 / T_c

        raw_rows.append({
            "condition": condition,
            "method": "raw_htfa",
            "T": T_c,
            "V": stack.shape[2],
            "chance": chance,
            "target_ratio": 1.0,
            "actual_ratio": 1.0,
            "K": stack.shape[2],
            **summary,
        })

    raw_df = pd.DataFrame(raw_rows)
    raw_df.to_csv(raw_csv, index=False)

else:
    raw_df = pd.read_csv(raw_csv)

display(raw_df)

In [ ]:
# ============================================================
# PCA — WITHIN CONDITION, NODE-SPACE
# ============================================================

pca_csv = OUTPUT_DIR / "pca_summary.csv"

if RUN_PCA:

    pca_rows = []

    # Use the requested ratio grid from spatial AA because both operate on V=700.
    pca_ratio_map = (
        ratio_map[
            ratio_map["analysis_type"] == "spatial"
        ][["condition", "target_ratio", "K", "actual_ratio"]]
        .drop_duplicates()
        .sort_values(["condition", "target_ratio"])
    )

    for condition in CONDITIONS:
        subjects = condition_data[condition]

        pooled = np.vstack(subjects)

        model = PCA(
            n_components=700,
            svd_solver="full",
            whiten=False,
            random_state=RNG_SEED,
        )

        model.fit(pooled)

        cond_rows = pca_ratio_map[
            pca_ratio_map["condition"] == condition
        ]

        for _, r in cond_rows.iterrows():
            K = int(r["K"])

            recon = []

            for X in subjects:
                Z = model.transform(X)[:, :K]
                Xhat = (
                    Z @ model.components_[:K, :]
                    + model.mean_
                )
                recon.append(
                    normalize_reconstruction(Xhat)
                )

            stack = np.stack(recon, axis=0)

            dec = run_timepoint_decoding(stack)
            summary = summarize_runs(dec)

            T_c = stack.shape[1]
            chance = 1.0 / T_c

            pca_rows.append({
                "condition": condition,
                "method": "PCA",
                "K": K,
                "target_ratio": float(r["target_ratio"]),
                "actual_ratio": K / 700,
                "T": T_c,
                "chance": chance,
                "variance_explained": float(
                    model.explained_variance_ratio_[:K].sum()
                ),
                **summary,
            })

    pca_df = pd.DataFrame(pca_rows)
    pca_df.to_csv(pca_csv, index=False)

else:
    pca_df = pd.read_csv(pca_csv)

display(pca_df.head(20))

In [ ]:
# ============================================================
# LOAD / RECONSTRUCT MS-AA
# ============================================================

def fit_path(analysis_type, condition, K):
    return (
        FIT_DIR
        / f"{analysis_type}AA_within_{condition}_K{K}.npz"
    )

def load_fit(analysis_type, condition, K):
    path = fit_path(
        analysis_type,
        condition,
        K,
    )

    if not path.exists():
        raise FileNotFoundError(path)

    data = np.load(
        path,
        allow_pickle=True,
    )

    results_subj = data[
        "results_subj"
    ].tolist()

    if isinstance(
        results_subj,
        np.ndarray,
    ):
        results_subj = results_subj.tolist()

    return {
        "results_subj": results_subj,
        "source": str(path),
    }

def reconstruct_from_archetypes(
    sub,
    analysis_type,
    archetype_indices=None,
):
    sXC = np.asarray(
        sub["sXC"],
        dtype=float,
    )

    S = np.asarray(
        sub["S"],
        dtype=float,
    )

    if archetype_indices is None:
        idx = np.arange(
            S.shape[0]
        )
    else:
        idx = np.asarray(
            list(archetype_indices),
            dtype=int,
        )

    if analysis_type == "spatial":
        Xhat = (
            sXC[:, idx]
            @ S[idx, :]
        )

    elif analysis_type == "temporal":
        Xhat = (
            sXC[:, idx]
            @ S[idx, :]
        ).T

    else:
        raise ValueError(
            "analysis_type must be spatial or temporal"
        )

    return normalize_reconstruction(Xhat)

def build_aa_stack(
    subjects,
    analysis_type,
    archetype_indices=None,
):
    return np.stack(
        [
            reconstruct_from_archetypes(
                sub,
                analysis_type,
                archetype_indices=archetype_indices,
            )
            for sub in subjects
        ],
        axis=0,
    )

In [ ]:
# ============================================================
# FULL MS-AA DECODING
# ============================================================

aa_full_csv = OUTPUT_DIR / "aa_full_summary.csv"

if RUN_AA_FULL:

    existing = (
        pd.read_csv(aa_full_csv)
        if aa_full_csv.exists()
        else pd.DataFrame()
    )

    rows = []

    for _, r in ratio_map.iterrows():

        analysis_type = str(r["analysis_type"])
        condition = str(r["condition"])
        K = int(r["K"])

        key_match = pd.Series(False, index=existing.index)

        if len(existing):
            key_match = (
                (existing["analysis_type"].astype(str) == analysis_type)
                & (existing["condition"].astype(str) == condition)
                & (existing["K"].astype(int) == K)
            )

        if SKIP_EXISTING and len(existing) and key_match.any():
            continue

        path = fit_path(
            analysis_type,
            condition,
            K,
        )

        if not path.exists():
            print("Missing fit; skipping:", path)
            continue

        print(
            "AA full:",
            analysis_type,
            condition,
            "K=", K,
            "ratio=", r["actual_ratio"],
        )

        fit = load_fit(
            analysis_type,
            condition,
            K,
        )

        stack = build_aa_stack(
            fit["results_subj"],
            analysis_type,
        )

        dec = run_timepoint_decoding(stack)
        summary = summarize_runs(dec)

        T_c = stack.shape[1]
        chance = 1.0 / T_c

        rows.append({
            "analysis_type": analysis_type,
            "condition": condition,
            "K": K,
            "target_ratio": float(r["target_ratio"]),
            "actual_ratio": float(r["actual_ratio"]),
            "ratio_denominator": int(r["denominator"]),
            "T": T_c,
            "chance": chance,
            "fit_source": fit["source"],
            **summary,
        })

    new = pd.DataFrame(rows)

    if len(existing):
        aa_full_df = pd.concat(
            [existing, new],
            ignore_index=True,
        )
        aa_full_df = (
            aa_full_df
            .sort_values(
                ["analysis_type", "condition", "K"]
            )
            .drop_duplicates(
                ["analysis_type", "condition", "K"],
                keep="last",
            )
        )
    else:
        aa_full_df = new

    aa_full_df.to_csv(
        aa_full_csv,
        index=False,
    )

else:
    aa_full_df = pd.read_csv(
        aa_full_csv
    )

display(aa_full_df.head(20))

## Optional paper-style top-\(m\) decoding

This section deliberately reproduces the **current manuscript's ranking logic** so the
four-condition version of Figures 3–5 can be generated. It is not the final fix for
the reviewer's circularity concern.

The clean paragraph-condition inference comes from the **full reconstruction / PCA /
raw HTFA analysis above**. For the final revised sparse-analysis claims, rerun top-\(m\)
selection with independent held-out subject ranking.

In [ ]:
# ============================================================
# SINGLE-ARCHETYPE + TOP-M
# ============================================================

per_arch_csv = OUTPUT_DIR / "aa_per_archetype_summary.csv"
topm_csv = OUTPUT_DIR / "aa_topm_summary.csv"

if RUN_AA_TOPM:

    if TOPM_SELECTION_MODE != "same_data":
        raise NotImplementedError(
            "This notebook currently implements manuscript-compatible "
            "same_data ranking only. Use a nested held-out ranking analysis "
            "for the confirmatory revision."
        )

    print(
        "\nWARNING: TOP-M SELECTION MODE = same_data\n"
        "This reproduces the current paper figures but does NOT resolve "
        "the reviewer's selection/evaluation dependence concern.\n"
    )

    existing_per = (
        pd.read_csv(per_arch_csv)
        if per_arch_csv.exists()
        else pd.DataFrame()
    )

    existing_topm = (
        pd.read_csv(topm_csv)
        if topm_csv.exists()
        else pd.DataFrame()
    )

    per_rows = []
    top_rows = []

    working = ratio_map.copy()

    if TOPM_TARGET_RATIOS is not None:
        working = working[
            working["target_ratio"].round(6).isin(
                [round(x, 6) for x in TOPM_TARGET_RATIOS]
            )
        ].copy()

    for _, r in working.iterrows():

        analysis_type = str(r["analysis_type"])
        condition = str(r["condition"])
        K = int(r["K"])

        path = fit_path(
            analysis_type,
            condition,
            K,
        )

        if not path.exists():
            continue

        fit = load_fit(
            analysis_type,
            condition,
            K,
        )

        subjects = fit["results_subj"]

        n_arch = int(
            np.asarray(
                subjects[0]["S"]
            ).shape[0]
        )

        scores = []

        for archetype in range(n_arch):

            cached = pd.DataFrame()

            if len(existing_per):
                cached = existing_per[
                    (existing_per["analysis_type"].astype(str) == analysis_type)
                    & (existing_per["condition"].astype(str) == condition)
                    & (existing_per["K"].astype(int) == K)
                    & (existing_per["archetype"].astype(int) == archetype)
                ]

            if SKIP_EXISTING and len(cached):
                mean_acc = float(
                    cached.iloc[-1]["mean"]
                )

            else:
                print(
                    f"single arch | {analysis_type} | {condition} | "
                    f"K={K} | a={archetype}"
                )

                stack = build_aa_stack(
                    subjects,
                    analysis_type,
                    archetype_indices=[archetype],
                )

                dec = run_timepoint_decoding(stack)
                summary = summarize_runs(dec)

                mean_acc = summary["mean"]

                per_rows.append({
                    "analysis_type": analysis_type,
                    "condition": condition,
                    "K": K,
                    "target_ratio": float(r["target_ratio"]),
                    "actual_ratio": float(r["actual_ratio"]),
                    "archetype": archetype,
                    **summary,
                })

            scores.append(
                (archetype, mean_acc)
            )

        ranking = [
            a for a, score in sorted(
                scores,
                key=lambda x: x[1],
                reverse=True,
            )
        ]

        for m in TOP_M_VALUES:

            if m > n_arch:
                continue

            cached = pd.DataFrame()

            if len(existing_topm):
                cached = existing_topm[
                    (existing_topm["analysis_type"].astype(str) == analysis_type)
                    & (existing_topm["condition"].astype(str) == condition)
                    & (existing_topm["K"].astype(int) == K)
                    & (existing_topm["top_m"].astype(int) == m)
                ]

            if SKIP_EXISTING and len(cached):
                continue

            selected = ranking[:m]

            stack = build_aa_stack(
                subjects,
                analysis_type,
                archetype_indices=selected,
            )

            dec = run_timepoint_decoding(stack)
            summary = summarize_runs(dec)

            T_c = stack.shape[1]
            chance = 1.0 / T_c

            top_rows.append({
                "analysis_type": analysis_type,
                "condition": condition,
                "K": K,
                "target_ratio": float(r["target_ratio"]),
                "actual_ratio": float(r["actual_ratio"]),
                "top_m": m,
                "selected_archetypes": str(selected),
                "T": T_c,
                "chance": chance,
                "selection_mode": TOPM_SELECTION_MODE,
                **summary,
            })

    if per_rows:
        existing_per = pd.concat(
            [existing_per, pd.DataFrame(per_rows)],
            ignore_index=True,
        )

        existing_per = (
            existing_per
            .sort_values(
                ["analysis_type", "condition", "K", "archetype"]
            )
            .drop_duplicates(
                ["analysis_type", "condition", "K", "archetype"],
                keep="last",
            )
        )

        existing_per.to_csv(
            per_arch_csv,
            index=False,
        )

    if top_rows:
        existing_topm = pd.concat(
            [existing_topm, pd.DataFrame(top_rows)],
            ignore_index=True,
        )

        existing_topm = (
            existing_topm
            .sort_values(
                ["analysis_type", "condition", "K", "top_m"]
            )
            .drop_duplicates(
                ["analysis_type", "condition", "K", "top_m"],
                keep="last",
            )
        )

        existing_topm.to_csv(
            topm_csv,
            index=False,
        )

    per_arch_df = existing_per
    topm_df = existing_topm

else:
    per_arch_df = (
        pd.read_csv(per_arch_csv)
        if per_arch_csv.exists()
        else pd.DataFrame()
    )
    topm_df = (
        pd.read_csv(topm_csv)
        if topm_csv.exists()
        else pd.DataFrame()
    )

print("Per-archetype rows:", len(per_arch_df))
print("Top-m rows:", len(topm_df))

if len(topm_df):
    display(topm_df.head(20))

In [ ]:
# ============================================================
# ADD CHANCE-CORRECTED / RAW-NORMALIZED METRICS
# ============================================================

raw_lookup = raw_df.set_index(
    "condition"
)["mean"].to_dict()

def add_retention_metrics(df):
    out = df.copy()

    out["above_chance"] = (
        out["mean"]
        - out["chance"]
    )

    out["raw_mean"] = out["condition"].map(
        raw_lookup
    )

    out["raw_above_chance"] = (
        out["raw_mean"]
        - out["chance"]
    )

    out["fraction_raw_above_chance_retained"] = (
        out["above_chance"]
        / out["raw_above_chance"]
    )

    return out

pca_df = add_retention_metrics(pca_df)
aa_full_df = add_retention_metrics(aa_full_df)

pca_df.to_csv(
    OUTPUT_DIR / "pca_summary_with_retention.csv",
    index=False,
)

aa_full_df.to_csv(
    OUTPUT_DIR / "aa_full_summary_with_retention.csv",
    index=False,
)

if len(topm_df):
    topm_df = add_retention_metrics(topm_df)

    topm_df.to_csv(
        OUTPUT_DIR / "aa_topm_summary_with_retention.csv",
        index=False,
    )

print("Saved decoding outputs to:", OUTPUT_DIR.resolve())